# 📊 Analisis Sentimen Ulasan Spotify — PyCaret AutoML
## SD25-32202 Pemrosesan Bahasa Alami — ITERA
### Checkpoint 2: Model Machine Learning dengan PyCaret

**Dataset:** Ulasan Aplikasi Spotify dari Google Play Store (100.000 data)  
**Tugas:** Klasifikasi Sentimen (Positif / Netral / Negatif)  
**Algoritma:** SVM, Multinomial Naive Bayes, Decision Tree  
**Framework:** PyCaret AutoML

## 1. Instalasi Library & Import

In [ ]:
# Install dependencies (jalankan di Google Colab)
!pip install pycaret[full] sastrawi wordcloud -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("✅ Import selesai!")

## 2. Load Dataset

In [ ]:
# Load dataset
# Sesuaikan path jika perlu
df = pd.read_csv('ulasan_com_spotify_music.csv')

print(f"Shape: {df.shape}")
print(f"\nKolom: {df.columns.tolist()}")
print(f"\nInfo:")
df.info()
print(f"\n5 Data Pertama:")
df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
# Statistik deskriptif
print("=" * 60)
print("STATISTIK DESKRIPTIF DATASET")
print("=" * 60)
print(f"Jumlah data       : {len(df):,}")
print(f"Jumlah kolom      : {df.shape[1]}")
print(f"Missing values    :")
print(df.isnull().sum().to_string())
print(f"\nDuplikat ulasan   : {df.duplicated(subset='Ulasan').sum():,}")
print(f"\nStatistik panjang ulasan (karakter):")
df['panjang_ulasan'] = df['Ulasan'].astype(str).str.len()
print(df['panjang_ulasan'].describe().to_string())

### 3.2 Distribusi Rating

In [ ]:
# Distribusi Rating
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
rating_counts = df['Rating'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71', '#27ae60']
axes[0].bar(rating_counts.index, rating_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Jumlah')
axes[0].set_title('Distribusi Rating Ulasan Spotify')
for i, v in enumerate(rating_counts.values):
    axes[0].text(rating_counts.index[i], v + 500, f'{v:,}', ha='center', fontweight='bold', fontsize=10)

# Pie chart
axes[1].pie(rating_counts.values, labels=[f'Rating {i}' for i in rating_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90, explode=[0.05]*5)
axes[1].set_title('Proporsi Rating')

plt.tight_layout()
plt.savefig('distribusi_rating.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nDistribusi Rating:\n{rating_counts.to_string()}")

### 3.3 Distribusi Panjang Ulasan

In [ ]:
# Distribusi panjang ulasan per rating
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
for rating in sorted(df['Rating'].unique()):
    subset = df[df['Rating'] == rating]['panjang_ulasan']
    axes[0].hist(subset, bins=50, alpha=0.5, label=f'Rating {rating}')
axes[0].set_xlabel('Panjang Ulasan (karakter)')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Panjang Ulasan per Rating')
axes[0].legend()
axes[0].set_xlim(0, 300)

# Boxplot
df.boxplot(column='panjang_ulasan', by='Rating', ax=axes[1])
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Panjang Ulasan (karakter)')
axes[1].set_title('Boxplot Panjang Ulasan per Rating')
plt.suptitle('')

plt.tight_layout()
plt.savefig('distribusi_panjang_ulasan.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 WordCloud

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sentimen_labels = {
    'Negatif': df[df['Rating'].isin([1, 2])],
    'Netral': df[df['Rating'] == 3],
    'Positif': df[df['Rating'].isin([4, 5])]
}

wc_colors = {'Negatif': 'Reds', 'Netral': 'Oranges', 'Positif': 'Greens'}

for idx, (label, data) in enumerate(sentimen_labels.items()):
    text = ' '.join(data['Ulasan'].astype(str).tolist())
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=wc_colors[label], max_words=100).generate(text)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(f'WordCloud — {label}', fontsize=14, fontweight='bold')
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('wordcloud_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Preprocessing Teks

### 4.1 Labeling Sentimen

In [ ]:
# Mapping Rating → Sentimen
def map_sentimen(rating):
    if rating <= 2:
        return 'Negatif'
    elif rating == 3:
        return 'Netral'
    else:
        return 'Positif'

df['Sentimen'] = df['Rating'].apply(map_sentimen)

print("Distribusi Sentimen:")
print(df['Sentimen'].value_counts())
print(f"\nProporsi:")
print((df['Sentimen'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

# Visualisasi
fig, ax = plt.subplots(figsize=(8, 5))
colors_sent = {'Positif': '#27ae60', 'Negatif': '#e74c3c', 'Netral': '#f39c12'}
sent_counts = df['Sentimen'].value_counts()
bars = ax.bar(sent_counts.index, sent_counts.values, 
              color=[colors_sent[s] for s in sent_counts.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, sent_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 500, f'{val:,}', ha='center', fontweight='bold')
ax.set_ylabel('Jumlah')
ax.set_title('Distribusi Sentimen (Sebelum Balancing)')
plt.tight_layout()
plt.savefig('distribusi_sentimen.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Text Cleaning & Preprocessing

In [ ]:
# ---- Kamus slang bahasa Indonesia (subset umum) ----
slang_dict = {
    'gk': 'tidak', 'ga': 'tidak', 'gak': 'tidak', 'g': 'tidak', 'ngga': 'tidak',
    'nggak': 'tidak', 'gabisa': 'tidak bisa', 'gaada': 'tidak ada', 'gada': 'tidak ada',
    'gbs': 'tidak bisa', 'gbsa': 'tidak bisa', 'tdk': 'tidak', 'gx': 'tidak',
    'engga': 'tidak', 'enggak': 'tidak', 'kagak': 'tidak', 'kaga': 'tidak',
    'udh': 'sudah', 'udah': 'sudah', 'uda': 'sudah', 'dah': 'sudah', 'sdh': 'sudah',
    'blm': 'belum', 'blum': 'belum', 'belom': 'belum',
    'bgt': 'banget', 'bngt': 'banget', 'bngtt': 'banget', 'bgtt': 'banget',
    'bkn': 'bukan', 'krn': 'karena', 'karna': 'karena', 'keren': 'keren',
    'tp': 'tapi', 'tpi': 'tapi', 'yg': 'yang', 'dg': 'dengan', 'dgn': 'dengan',
    'sm': 'sama', 'sma': 'sama', 'utk': 'untuk', 'u/': 'untuk', 'bwt': 'buat',
    'jg': 'juga', 'jga': 'juga', 'jgn': 'jangan', 'jngn': 'jangan',
    'knp': 'kenapa', 'gmn': 'bagaimana', 'gmna': 'bagaimana',
    'emg': 'memang', 'emang': 'memang', 'mmg': 'memang',
    'lg': 'lagi', 'lgi': 'lagi', 'byk': 'banyak', 'bnyk': 'banyak',
    'bgus': 'bagus', 'bgs': 'bagus', 'jelek': 'jelek', 'jlk': 'jelek',
    'mantap': 'mantap', 'mantep': 'mantap', 'mntap': 'mantap', 'mantab': 'mantap',
    'aj': 'saja', 'aja': 'saja', 'doang': 'saja', 'doank': 'saja',
    'bs': 'bisa', 'bsa': 'bisa', 'bls': 'balas',
    'smga': 'semoga', 'smoga': 'semoga', 'moga': 'semoga',
    'hrs': 'harus', 'hrus': 'harus', 'msh': 'masih', 'msih': 'masih',
    'skrg': 'sekarang', 'skrang': 'sekarang', 'skg': 'sekarang',
    'org': 'orang', 'orng': 'orang',
    'app': 'aplikasi', 'apl': 'aplikasi', 'apk': 'aplikasi', 'appnya': 'aplikasinya',
    'lagu2': 'lagu lagu', 'lagu²': 'lagu lagu',
    'pgn': 'ingin', 'pengen': 'ingin', 'pngen': 'ingin',
    'bnr': 'benar', 'bner': 'benar',
    'krg': 'kurang', 'kurng': 'kurang',
    'req': 'request', 'tlg': 'tolong', 'tlong': 'tolong', 'tolong': 'tolong',
    'mkin': 'makin', 'mkn': 'makin', 'smakin': 'semakin',
    'dl': 'dulu', 'dlu': 'dulu',
    'gpp': 'tidak apa-apa', 'gapapa': 'tidak apa-apa',
    'dngerin': 'mendengarkan', 'dengerin': 'mendengarkan', 'dnger': 'dengar',
    'ngebug': 'bug', 'lemot': 'lambat', 'lelet': 'lambat',
    'njir': '', 'anjir': '', 'wkwk': '', 'wkwkwk': '', 'haha': '', 'hehe': '',
    'btw': '', 'cmn': 'cuma', 'cuma': 'cuma', 'cuman': 'cuma',
}

print(f"Jumlah kata dalam kamus slang: {len(slang_dict)}")

In [ ]:
# ---- Fungsi Preprocessing ----
# Inisialisasi Sastrawi
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.createStemmer()

stopword_factory = StopWordRemoverFactory()
stopwords_id = set(stopword_factory.getStopWords())
# Tambahkan stopwords khusus
stopwords_id.update(['yg', 'nya', 'di', 'ke', 'dr', 'utk', 'ya', 'nih', 'sih', 'dong', 'deh',
                     'loh', 'kan', 'kok', 'tuh', 'nah', 'lah', 'kali', 'dong', 'banget',
                     'spotify', 'aplikasi', 'app', 'music', 'lagu'])

def clean_text(text):
    """Pipeline preprocessing teks"""
    if not isinstance(text, str):
        return ''
    
    text = text.lower()  # Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)  # Hapus URL
    text = re.sub(r'@\w+', '', text)  # Hapus mention
    text = re.sub(r'#\w+', '', text)  # Hapus hashtag
    text = re.sub(r'[^\w\s]', ' ', text)  # Hapus tanda baca & emoji
    text = re.sub(r'\d+', '', text)  # Hapus angka
    
    # Normalisasi slang
    words = text.split()
    words = [slang_dict.get(w, w) for w in words]
    
    # Hapus stopwords
    words = [w for w in words if w not in stopwords_id and len(w) > 1]
    
    # Stemming
    words = [stemmer.stem(w) for w in words]
    
    # Gabungkan kembali
    text = ' '.join(words)
    text = re.sub(r'\s+', ' ', text).strip()  # Hapus spasi berlebih
    
    return text

# Test preprocessing
test_texts = [
    "suka banget sama ni app",
    "ga bisa denger lagu JKT48 lagi gara² selalu di tanya mau premium gak👊🏻",
    "Mantap tapi banyak iklan",
    "najis ceo nya GK punya empati, udh itu iklan Mulu"
]

print("Contoh Hasil Preprocessing:")
print("-" * 60)
for t in test_texts:
    print(f"ASLI   : {t}")
    print(f"BERSIH : {clean_text(t)}")
    print()

In [ ]:
# ---- Terapkan Preprocessing ke Seluruh Dataset ----
from tqdm import tqdm
tqdm.pandas(desc="Preprocessing")

print("Memulai preprocessing 100.000 ulasan...")
print("⚠️ Proses ini memakan waktu ~15-30 menit karena stemming Sastrawi.")
print("   Jika terlalu lama, pertimbangkan sampling atau skip stemming.\n")

df['ulasan_bersih'] = df['Ulasan'].progress_apply(clean_text)

# Hapus baris kosong setelah preprocessing
before = len(df)
df = df[df['ulasan_bersih'].str.strip() != ''].reset_index(drop=True)
after = len(df)
print(f"\nData sebelum: {before:,} | Sesudah: {after:,} | Dihapus: {before - after:,}")
print("\nSample hasil preprocessing:")
df[['Ulasan', 'ulasan_bersih', 'Sentimen']].sample(5, random_state=42)

## 5. Model Machine Learning dengan PyCaret

### 5.1 Setup PyCaret

In [ ]:
from pycaret.classification import *

# Siapkan data untuk PyCaret
df_model = df[['ulasan_bersih', 'Sentimen']].copy()
df_model.columns = ['text', 'label']

print(f"Data untuk modeling: {df_model.shape}")
print(f"\nDistribusi label:\n{df_model['label'].value_counts()}")

In [ ]:
# Setup PyCaret
clf = setup(
    data=df_model,
    target='label',
    text_features=['text'],           # PyCaret akan otomatis TF-IDF
    fix_imbalance=True,               # SMOTE untuk handle imbalanced data
    fix_imbalance_method=None,        # Default: SMOTE
    session_id=42,
    verbose=True,
    fold=5,                           # 5-fold cross validation
    train_size=0.8,
    html=False,
    log_experiment=False
)

print("\n✅ PyCaret setup selesai!")

### 5.2 Benchmark 3 Algoritma ML

In [ ]:
# Bandingkan 3 model yang dipilih: SVM, Multinomial NB, Decision Tree
# PyCaret model IDs:
#   - 'svm'  : Support Vector Machine (linear)
#   - 'nb'   : Naive Bayes
#   - 'dt'   : Decision Tree

best_models = compare_models(
    include=['svm', 'nb', 'dt'],
    sort='F1',
    n_select=3,
    cross_validation=True,
    fold=5
)

print("\n✅ Perbandingan selesai!")

### 5.3 Evaluasi Detail Setiap Model

In [ ]:
# ---- Buat dan evaluasi masing-masing model ----
model_names = {
    'svm': 'Support Vector Machine',
    'nb': 'Naive Bayes (Multinomial)',
    'dt': 'Decision Tree'
}

trained_models = {}
results_list = []

for model_id, model_name in model_names.items():
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"{'='*60}")
    
    # Create model
    model = create_model(model_id, fold=5)
    trained_models[model_id] = model
    
    # Get cross-validation results
    results = pull()
    
    # Ambil rata-rata metrik (baris 'Mean')
    mean_row = results.iloc[-2]  # Mean row
    results_list.append({
        'Model': model_name,
        'Accuracy': round(mean_row['Accuracy'], 4),
        'AUC': round(mean_row.get('AUC', 0), 4),
        'Recall': round(mean_row['Recall'], 4),
        'Precision': round(mean_row['Prec.'], 4),
        'F1-Score': round(mean_row['F1'], 4),
        'Kappa': round(mean_row['Kappa'], 4)
    })
    
    print(f"\n📊 {model_name} — Evaluasi:")
    print(f"   Accuracy  : {results_list[-1]['Accuracy']}")
    print(f"   Precision : {results_list[-1]['Precision']}")
    print(f"   Recall    : {results_list[-1]['Recall']}")
    print(f"   F1-Score  : {results_list[-1]['F1-Score']}")

print("\n✅ Semua model selesai di-training!")

In [ ]:
# ---- Tabel Perbandingan ----
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("  TABEL PERBANDINGAN PERFORMA 3 MODEL ML")
print("=" * 70)
print(results_df.to_string(index=False))

# Highlight best
best = results_df.iloc[0]
print(f"\n🏆 Model Terbaik: {best['Model']} (F1-Score: {best['F1-Score']})")

In [ ]:
# ---- Visualisasi Perbandingan ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart perbandingan metrik
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(metrics))
width = 0.25
colors_model = ['#3498db', '#e74c3c', '#2ecc71']

for i, (_, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics]
    axes[0].bar(x + i * width, vals, width, label=row['Model'], color=colors_model[i], edgecolor='white')

axes[0].set_xlabel('Metrik')
axes[0].set_ylabel('Skor')
axes[0].set_title('Perbandingan Metrik Evaluasi — 3 Model ML')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)

# Radar chart / spider chart
from matplotlib.patches import FancyBboxPatch
metrics_radar = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Kappa']
angles = np.linspace(0, 2 * np.pi, len(metrics_radar), endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(122, polar=True)
for i, (_, row) in enumerate(results_df.iterrows()):
    vals = [row[m] for m in metrics_radar]
    vals += vals[:1]
    ax_radar.plot(angles, vals, 'o-', linewidth=2, label=row['Model'], color=colors_model[i])
    ax_radar.fill(angles, vals, alpha=0.1, color=colors_model[i])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metrics_radar)
ax_radar.set_ylim(0, 1)
ax_radar.set_title('Radar Chart Performa Model', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('perbandingan_model.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.4 Confusion Matrix & Classification Report

In [ ]:
# Confusion Matrix untuk setiap model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (model_id, model_name) in enumerate(model_names.items()):
    model = trained_models[model_id]
    
    # Plot confusion matrix menggunakan PyCaret
    print(f"\n{'='*50}")
    print(f"Classification Report — {model_name}")
    print(f"{'='*50}")
    
    # Evaluate on test set
    plot_model(model, plot='confusion_matrix', save=True)
    
    # Classification report
    plot_model(model, plot='class_report', save=True)

print("\n✅ Confusion matrix dan classification report tersimpan!")

## 6. Finalisasi & Simpan Model Terbaik

### 6.1 Pilih dan Tune Model Terbaik

In [ ]:
# Identifikasi model terbaik berdasarkan F1-Score
best_model_name = results_df.iloc[0]['Model']
best_model_id = [k for k, v in model_names.items() if v == best_model_name][0]

print(f"Model terbaik: {best_model_name} ({best_model_id})")
print(f"F1-Score: {results_df.iloc[0]['F1-Score']}")

# Tuning hyperparameter model terbaik
print(f"\nTuning hyperparameter {best_model_name}...")
tuned_model = tune_model(
    trained_models[best_model_id],
    fold=5,
    optimize='F1',
    n_iter=50
)

# Hasil setelah tuning
tuned_results = pull()
print(f"\nHasil setelah tuning:")
print(tuned_results)

### 6.2 Finalisasi & Simpan Model

In [ ]:
# Finalisasi model (train pada seluruh data)
final_model = finalize_model(tuned_model)

# Simpan model
save_model(final_model, 'model_sentimen_spotify_ml')
print("\n✅ Model tersimpan sebagai 'model_sentimen_spotify_ml.pkl'")

# Simpan juga pipeline preprocessing info
import pickle
preprocessing_info = {
    'slang_dict': slang_dict,
    'stopwords': list(stopwords_id),
    'sentiment_mapping': {1: 'Negatif', 2: 'Negatif', 3: 'Netral', 4: 'Positif', 5: 'Positif'},
    'model_name': best_model_name,
    'metrics': results_df.to_dict('records')
}
with open('preprocessing_info.pkl', 'wb') as f:
    pickle.dump(preprocessing_info, f)

print("✅ Info preprocessing tersimpan sebagai 'preprocessing_info.pkl'")

### 6.3 Test Prediksi

In [ ]:
# Test prediksi pada teks baru
test_reviews = [
    "aplikasi ini sangat bagus dan membantu",
    "banyak iklan sangat mengganggu tidak bisa mendengarkan lagu",
    "lumayan lah tapi masih perlu perbaikan",
    "suka banget sama spotify premium",
    "error terus tidak bisa dibuka sama sekali",
    "biasa aja sih standar"
]

test_df = pd.DataFrame({'text': [clean_text(t) for t in test_reviews]})

predictions = predict_model(final_model, data=test_df)

print("\n" + "=" * 70)
print("  TEST PREDIKSI SENTIMEN")
print("=" * 70)
for orig, pred_label in zip(test_reviews, predictions['prediction_label']):
    emoji = {'Positif': '😊', 'Negatif': '😠', 'Netral': '😐'}.get(pred_label, '❓')
    print(f"  {emoji} [{pred_label:^8}] {orig}")

## 7. Simpan Hasil untuk Paper

In [ ]:
# Simpan tabel hasil ke CSV untuk digunakan di paper
results_df.to_csv('hasil_benchmark_ml.csv', index=False)
print("✅ Hasil benchmark tersimpan ke 'hasil_benchmark_ml.csv'")
print(f"\nRingkasan Eksperimen:")
print(f"  Dataset     : Ulasan Spotify Google Play Store")
print(f"  Jumlah Data : {len(df):,}")
print(f"  Kelas       : 3 (Positif, Netral, Negatif)")
print(f"  Balancing   : SMOTE (via PyCaret fix_imbalance)")
print(f"  Feature     : TF-IDF (otomatis oleh PyCaret)")
print(f"  Algoritma   : SVM, Naive Bayes, Decision Tree")
print(f"  Best Model  : {best_model_name}")
print(f"  Best F1     : {results_df.iloc[0]['F1-Score']}")

print(f"\n📁 File yang dihasilkan:")
print(f"  - model_sentimen_spotify_ml.pkl")
print(f"  - preprocessing_info.pkl")
print(f"  - hasil_benchmark_ml.csv")
print(f"  - distribusi_rating.png")
print(f"  - distribusi_sentimen.png")
print(f"  - distribusi_panjang_ulasan.png")
print(f"  - wordcloud_sentimen.png")
print(f"  - perbandingan_model.png")